In [49]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
import sys
 
sys.path.append('/Users/irfan.hilman/ai-am')

from function import utils


In [50]:
df = pd.read_csv('Database_2016-01-01_2026-08-28.csv', index_col=0)
df = utils.add_momentum(data=df, period=5, skip_days=0)
df = utils.add_momentum(data=df, period=10, skip_days=5)
df = utils.add_momentum(data=df, period=15, skip_days=10)
df = utils.add_momentum(data=df, period=20, skip_days=15)
df = utils.add_momentum(data=df, period=40, skip_days=20)
df = utils.add_momentum(data=df, period=60, skip_days=40)
df = utils.rolling_bwd_param(data=df, type='Median', param='Transaction Value', period=20)
df = utils.rolling_bwd_param(data=df, type='Median', param='Volume', period=20)
df = utils.rolling_bwd_param(data=df, type='Median', param='Market Cap', period=20)

df = utils.add_forward_return(data=df, horizon=10)
n_decile = 9
df['FwdReturn 10d Decile'] = df.groupby('Date')['FwdReturn 10d'].transform(utils.assign_decile, n_deciles=n_decile)
df = df.sort_values(by=['Date','Kode'])
df = df.reset_index(drop=True)

df = df[['Date','Kode','Momentum 5d-0d','Momentum 10d-5d','Momentum 15d-10d','Momentum 20d-15d','Momentum 40d-20d','Momentum 60d-40d','FwdReturn 10d Decile']]

MODEL TRAINING

In [51]:
data = df.copy()
data['Date'] = pd.to_datetime(data['Date'])

feature_cols = [
    'Momentum 5d-0d', 'Momentum 10d-5d', 'Momentum 15d-10d',
    'Momentum 20d-15d', 'Momentum 40d-20d', 'Momentum 60d-40d']

target_col = 'FwdReturn 10d Decile'

model_df = data.dropna(subset=[target_col]).copy()
model_df = model_df.dropna(subset=feature_cols, how='any')
model_df['label'] = model_df[target_col].astype(int) - 1  # 0-7 for XGBoost


model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=8,
    random_state=42,
    n_jobs=-1)

model.fit(model_df[feature_cols], model_df['label'])

# Save
model.save_model('xgboost_model.json')
print("Model saved.")

Model saved.


In [ ]:
model = XGBClassifier()
model.load_model('xgboost_model.json')

# new_data = dataframe with the same feature_cols, e.g. latest date's stocks
new_data = data[data['Date'] == data['Date'].max()].copy()
new_data = new_data.dropna(subset=feature_cols, how='any')


probs = model.predict_proba(new_data[feature_cols])  # shape: (n_rows, 9)

# probs[:, 8] = probability of decile 9 (label 8 = highest decile)
new_data['Prob Decile 9'] = probs[:, 8]

result = new_data[['Date', 'Kode', 'Prob Decile 9']].sort_values('Prob Decile 9', ascending=False)
result

,Date,Kode,Prob Decile 9
2007065,2026-08-28,PACK,0.191441
2006699,2026-08-28,EKAD,0.152617
2006886,2026-08-28,KETR,0.146720
2007173,2026-08-28,SAFE,0.136357
2006980,2026-08-28,MGLV,0.131458
...,...,...,...
2007100,2026-08-28,PNBS,0.007871
2006954,2026-08-28,MASB,0.007614
2006657,2026-08-28,DADA,0.005861
2006487,2026-08-28,ATLA,0.005861


STOCK SELECTION

In [60]:
selected_stocks = list(result.head(10)['Kode'])

In [61]:
selected_stocks

['PACK',
 'EKAD',
 'KETR',
 'SAFE',
 'MGLV',
 'TMPO',
 'BAIK',
 'MDIA',
 'PICO',
 'NICE']